# State-space figures — 2-unit GRU (v8)

Five figures, all in the **2D hidden-state space** (hidden unit 1 vs hidden unit 2):

1. **All twelve subtasks, averaged** — one mean trajectory per subtask (averaged over its 100 test trials);
   each conflict subtask split by the network's output label, giving six conflict trajectories.
2. **Conflict subtasks on their own** — the same conflict trajectories isolated.
3. **Conflict by stimulus strength** — localisation-conflict trajectories coloured by the intensity gap
   between the two cues, so reliability-weighting is visible as the endpoint shifting with the gap.
4. **Variance across seeds** — for one subtask, the averaged trajectory of each of the five seeds overlaid.
5. **Decision boundaries across seeds** — the readout boundaries (lines only, no fill) for the five seeds overlaid.

Trains five 2-unit unified GRUs (seeds 0–4) on `./generated_trials_v8` and keeps all of them.

## 1. Setup, data, model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cpu")
DATA_DIR = Path("./generated_trials_v8")
OUT_DIR = Path("./statespace_v8"); OUT_DIR.mkdir(exist_ok=True)

def load(split):
    d = np.load(DATA_DIR / f"{split}.npz", allow_pickle=True)
    out = {"X": d["X"].astype(np.float32), "y": d["y_4class"].astype(np.int64), "types": d["types"]}
    if "aud_int" in d:
        out["aud_int"] = d["aud_int"].astype(np.float32); out["vis_int"] = d["vis_int"].astype(np.float32)
    return out
train, test = load("train"), load("test")

DET_SUBTASKS = ["det_absent", "det_auditory_only", "det_visual_only", "det_multisensory"]
LOC_SUBTASKS = ["loc_auditory_only_L", "loc_auditory_only_R", "loc_visual_only_L", "loc_visual_only_R",
                "loc_multisensory_same_L", "loc_multisensory_same_R",
                "loc_conflict_audL_visR", "loc_conflict_audR_visL"]
SUBTASKS = DET_SUBTASKS + LOC_SUBTASKS
CONFLICT_SUBTASKS = ["det_multisensory", "loc_conflict_audL_visR", "loc_conflict_audR_visL"]
LOC_CONFLICTS = ["loc_conflict_audL_visR", "loc_conflict_audR_visL"]
NONCONF = [s for s in SUBTASKS if s not in CONFLICT_SUBTASKS]
CLASS_NAMES  = ["no det (0)", "det (1)", "right (2)", "left (3)"]
CLASS_SHORT  = {0: "no-det", 1: "det", 2: "right", 3: "left"}
CLASS_COLORS = ["#cfcfcf", "#e0852e", "#c0392b", "#2e6ca4"]

_det_cols = plt.cm.Oranges(np.linspace(0.45, 0.92, len(DET_SUBTASKS)))
_loc_cols = plt.cm.Blues(np.linspace(0.35, 0.95, len(LOC_SUBTASKS)))
SUBTASK_COLOR = {}
for i, s in enumerate(DET_SUBTASKS): SUBTASK_COLOR[s] = _det_cols[i]
for i, s in enumerate(LOC_SUBTASKS): SUBTASK_COLOR[s] = _loc_cols[i]

class UnifiedGRU(nn.Module):
    def __init__(self, n_channels=4, hidden_size=2, n_classes=4):
        super().__init__()
        self.gru = nn.GRU(n_channels, hidden_size, batch_first=True)
        self.readout = nn.Linear(hidden_size, n_classes)
    def forward(self, x):
        x = x.transpose(1, 2); h, _ = self.gru(x); return self.readout(h)
HIDDEN = 2
print("Train:", train["X"].shape, " Test:", test["X"].shape)

## 2. Train five seeds and keep them all

In [ ]:
def train_model(seed, n_epochs=50, lr=1e-3, batch=64):
    torch.manual_seed(seed); np.random.seed(seed)
    model = UnifiedGRU(4, HIDDEN, 4).to(device)
    loader = DataLoader(TensorDataset(torch.from_numpy(train["X"]), torch.from_numpy(train["y"])),
                        batch_size=batch, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr); loss_fn = nn.CrossEntropyLoss()
    for _ in range(n_epochs):
        model.train()
        for Xb, yb in loader:
            logits = model(Xb); B, T, C = logits.shape
            loss = loss_fn(logits.reshape(B*T, C), yb.unsqueeze(1).expand(B, T).reshape(B*T))
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def nonconf_acc(model):
    model.eval()
    with torch.no_grad():
        pred = model(torch.from_numpy(test["X"]))[:, -1, :].argmax(-1).numpy()
    return np.mean([ (pred[test["types"]==s]==test["y"][test["types"]==s]).mean() for s in NONCONF])

SEEDS = [0, 1, 2, 3, 4]
models = []
for s in SEEDS:
    m = train_model(s); a = nonconf_acc(m); models.append(m)
    print("seed %d  non-conflict %.3f" % (s, a))
rep_idx = int(np.argmax([nonconf_acc(m) for m in models]))
rep_model = models[rep_idx]
print("representative seed for single-network figures:", SEEDS[rep_idx])

## 3. Helpers (hidden-state trajectories and decision regions)

In [ ]:
def hidden_states(model, X):
    model.eval()
    with torch.no_grad():
        h, _ = model.gru(torch.from_numpy(X).transpose(1, 2))   # (B, T, H)
    return h.numpy()

def mean_traj(model, X_subset):
    H = hidden_states(model, X_subset)         # (B, T, 2)
    m = H.mean(0)                              # (T, 2)
    return np.vstack([np.zeros((1, 2)), m])    # prepend the t=0 origin -> (T+1, 2)

def final_class(model, X_subset):
    H = hidden_states(model, X_subset)
    W = model.readout.weight.detach().numpy(); b = model.readout.bias.detach().numpy()
    return np.argmax(H[:, -1, :] @ W.T + b, axis=1)

GRID = np.linspace(-1.05, 1.05, 400)
GX, GY = np.meshgrid(GRID, GRID)
_flat = np.stack([GX.ravel(), GY.ravel()], 1)

def region_map(model):
    W = model.readout.weight.detach().numpy(); b = model.readout.bias.detach().numpy()
    return np.argmax(_flat @ W.T + b, axis=1).reshape(GX.shape)

def draw_regions(ax, model, alpha=0.18):
    ax.pcolormesh(GX, GY, region_map(model), cmap=ListedColormap(CLASS_COLORS),
                  alpha=alpha, shading="auto", vmin=0, vmax=3)
    ax.set_xlabel("hidden unit 1"); ax.set_ylabel("hidden unit 2")
    ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05); ax.set_aspect("equal")

def mark_ends(ax, tr, color):
    ax.scatter(tr[0, 0], tr[0, 1], color="k", s=18, zorder=5)                 # start (origin)
    ax.scatter(tr[-1, 0], tr[-1, 1], color=color, s=55, marker="*",
               edgecolor="k", lw=0.4, zorder=6)                               # end

## Figure 1. All twelve subtasks, averaged trajectories

Each non-conflict subtask is a single mean trajectory over its 100 trials. Each conflict subtask is split by
the network's output label and averaged within each group, so the conflicts appear as separate lines.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 8))
draw_regions(ax, rep_model)
# non-conflict subtasks: one averaged line each
for s in NONCONF:
    idx = np.where(test["types"] == s)[0]
    tr = mean_traj(rep_model, test["X"][idx])
    ax.plot(tr[:, 0], tr[:, 1], color=SUBTASK_COLOR[s], lw=1.8, label=s)
    mark_ends(ax, tr, SUBTASK_COLOR[s])
# conflict subtasks: split by predicted output label
conf_palette = plt.cm.tab10(np.linspace(0, 1, 10))
ci = 0
for s in CONFLICT_SUBTASKS:
    idx = np.where(test["types"] == s)[0]
    cls = final_class(rep_model, test["X"][idx])
    for c in np.unique(cls):
        sub = idx[cls == c]
        if len(sub) < 5:
            continue
        tr = mean_traj(rep_model, test["X"][sub])
        col = conf_palette[ci % 10]; ci += 1
        ax.plot(tr[:, 0], tr[:, 1], color=col, lw=2.4, ls="--",
                label="%s -> %s" % (s, CLASS_SHORT[c]))
        mark_ends(ax, tr, col)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=7.5, frameon=False)
ax.set_title("Average state-space trajectory per subtask (conflicts split by output label)")
plt.tight_layout(); plt.show()

## Figure 2. Conflict subtasks on their own

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7.5))
draw_regions(ax, rep_model)
ci = 0
for s in CONFLICT_SUBTASKS:
    idx = np.where(test["types"] == s)[0]
    cls = final_class(rep_model, test["X"][idx])
    for c in np.unique(cls):
        sub = idx[cls == c]
        if len(sub) < 5:
            continue
        tr = mean_traj(rep_model, test["X"][sub])
        col = conf_palette[ci % 10]; ci += 1
        ax.plot(tr[:, 0], tr[:, 1], color=col, lw=2.6, ls="--",
                label="%s -> %s  (n=%d)" % (s, CLASS_SHORT[c], len(sub)))
        mark_ends(ax, tr, col)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8, frameon=False)
ax.set_title("Conflict trajectories only, split by output label")
plt.tight_layout(); plt.show()

## Figure 3. Localisation conflict by stimulus strength

For each localisation conflict, trials are binned by the intensity gap between the two cues and averaged
within each bin. The colour gradient runs from a small gap (near-tie) to a large gap (one cue dominant).

In [ ]:
N_BINS = 5
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
for ax, s in zip(axes, LOC_CONFLICTS):
    draw_regions(ax, rep_model)
    idx = np.where(test["types"] == s)[0]
    gap = np.abs(test["aud_int"][idx] - test["vis_int"][idx])
    edges = np.quantile(gap, np.linspace(0, 1, N_BINS + 1)); edges[-1] += 1e-6
    cmap = plt.cm.viridis
    for b in range(N_BINS):
        sel = idx[(gap >= edges[b]) & (gap < edges[b + 1])]
        if len(sel) < 3:
            continue
        tr = mean_traj(rep_model, test["X"][sel])
        col = cmap(b / (N_BINS - 1))
        ax.plot(tr[:, 0], tr[:, 1], color=col, lw=2.2)
        mark_ends(ax, tr, col)
    ax.set_title(s)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(gap.min(), gap.max()))
    fig.colorbar(sm, ax=ax, label="intensity gap |stronger - weaker|", fraction=0.046)
fig.suptitle("Localisation conflict trajectories, coloured by intensity gap", fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.96]); plt.show()

## Figure 4. Variance across seeds (one subtask per plot)

For a chosen subtask, the averaged trajectory of each of the five seeds is overlaid. This shows how
consistent the trained networks are. Note the basis caveat below.

In [ ]:
SUBTASKS_TO_COMPARE = ["det_auditory_only", "loc_auditory_only_L", "loc_conflict_audL_visR"]
seed_cols = plt.cm.tab10(np.linspace(0, 1, len(SEEDS)))
fig, axes = plt.subplots(1, len(SUBTASKS_TO_COMPARE), figsize=(6 * len(SUBTASKS_TO_COMPARE), 6))
if len(SUBTASKS_TO_COMPARE) == 1: axes = [axes]
for ax, s in zip(axes, SUBTASKS_TO_COMPARE):
    idx = np.where(test["types"] == s)[0]
    for si, m in enumerate(models):
        tr = mean_traj(m, test["X"][idx])
        ax.plot(tr[:, 0], tr[:, 1], color=seed_cols[si], lw=1.8, label="seed %d" % SEEDS[si])
        ax.scatter(tr[-1, 0], tr[-1, 1], color=seed_cols[si], s=45, marker="*", edgecolor="k", lw=0.4, zorder=6)
    ax.scatter(0, 0, color="k", s=18, zorder=5)
    ax.set_xlabel("hidden unit 1"); ax.set_ylabel("hidden unit 2"); ax.set_aspect("equal")
    ax.set_title(s); ax.legend(fontsize=8, frameon=False)
fig.suptitle("Average trajectory per seed (one subtask each)", fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.show()

## Figure 5. Decision boundaries across seeds (lines only, overlaid)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 7.5))
for si, m in enumerate(models):
    reg = region_map(m).astype(float)
    ax.contour(GX, GY, reg, levels=[0.5, 1.5, 2.5], colors=[seed_cols[si]], linewidths=1.4)
    ax.plot([], [], color=seed_cols[si], lw=1.4, label="seed %d" % SEEDS[si])   # legend proxy
ax.set_xlabel("hidden unit 1"); ax.set_ylabel("hidden unit 2")
ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05); ax.set_aspect("equal")
ax.legend(fontsize=8, frameon=False, loc="upper left")
ax.set_title("Readout decision boundaries across seeds (lines only)")
plt.tight_layout(); plt.show()

## Note on comparing across seeds (figures 4 and 5)

Each network has its **own 2D basis**: the two hidden units are only defined up to rotation, reflection and
scaling, so two seeds can implement the *same* computation yet look different when their raw trajectories or
boundaries are overlaid. Large apparent differences in figures 4 and 5 should therefore be treated with care,
as they may be a change of basis rather than a change of behaviour. Procrustes-aligning the seeds' states
before overlaying (in the spirit of Maheswaranathan et al. 2019) would be the cleaner comparison.

Saving the trained seeds for re-use:

In [ ]:
for si, m in enumerate(models):
    torch.save(m.state_dict(), OUT_DIR / ("unified_h2_seed%d.pt" % SEEDS[si]))
print("saved 5 models to", OUT_DIR)